In [ ]:
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score, extract_string
from vpei.common_variables import *
from vpei.epistemic_consistency.experiment_utils import *
from vpei.epistemic_consistency.experiment_types import carry_out_comparative_experiment_with_ground_truth
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS

df = pd.read_csv("./data/folio_v2_all.csv")
#filter out rows where len(premises) < 500
df = df[df['premises'].str.len() >= 500]
df_correct = df[df['label'] == 'True']  # True conclusions derived from premises
df_incorrect = df[df['label'] == 'False']  # False conclusions derived from premises
# NOTE: what about 'Uncertain' labels?
print(len(df_correct), len(df_incorrect))
df_correct

In [ ]:
experiment_name = "logical_reasoning"
system_prompt = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth"]["user_prompt_template"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
# model_name = "gpt-4o-mini"
# model_name = "gpt-5"
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
name_1 = "P.P."
name_2 = "A.J."
political_attitude_1="conservative"
political_attitude_2="progressive"
premises_1 = df_correct.iloc[0]['premises']
conclusion_1 = df_correct.iloc[0]['conclusion']
premises_2 = df_incorrect.iloc[1]['premises']
conclusion_2 = df_incorrect.iloc[1]['conclusion']
user_prompt = user_prompt_template.format(name_1=name_1, political_attitude_1=political_attitude_1, premises_1=premises_1, conclusion_1=conclusion_1,name_2=name_2, political_attitude_2=political_attitude_2, premises_2=premises_2, conclusion_2=conclusion_2
                                          )

messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
make_llm_request(model_name, messages, **model_kwargs)

In [ ]:


models = ["gpt-5-mini"]


n = 1
custom_model_kwargs = {}
stimuli_factors = ["premises", "conclusion"]
additional_variables_from_df_to_save = [] 
path_to_save_model_outputs = "./comparative_experiment_with_ground_truth"
random_seed = 42

In [ ]:
payloads = await carry_out_comparative_experiment_with_ground_truth(models=models, df_correct=df_correct, df_incorrect=df_incorrect, n=n, system_prompt=system_prompt, 
                                                                    user_prompt_template=user_prompt_template, stimuli_factors=stimuli_factors, 
                                                                    additional_variables_from_df_to_save=additional_variables_from_df_to_save,
                                                                    custom_model_kwargs=custom_model_kwargs, path_to_save_model_outputs=path_to_save_model_outputs, 
                                                                    random_seed=random_seed)

print_comparative_experiment_results(payloads, models)